# 🏗️ Notebook 06: Transformer Architecture

This notebook assembles a complete transformer block from the components we built in previous notebooks: multi-head attention, feed-forward networks, layer normalization, and residual connections.

**Prerequisites:** Notebooks 01-05

**What you'll learn:**
1. Understand how attention, FFN, and normalization combine into a transformer block\n2. Compare activation functions (ReLU, GELU, SiLU, SwiGLU)\n3. Understand pre-norm vs post-norm and why pre-norm wins\n4. Count parameters in a transformer and estimate memory usage

💡 **In simple terms**, think of it as building blocks — each concept in this notebook is a building block that connects to the next. For example, understanding the basics here will make everything that follows much easier to grasp.

## ✅ Environment Validation

### 🧠 The Big Picture

Think of a transformer block like a factory assembly line — each station (attention, FFN, normalization) does one specific job, and the residual connections are like conveyor belts that carry the original material alongside the processed version, so nothing important gets lost along the way.

### 💡 Why Does This Matter?

Why do we need normalization? Without it, activations can grow or shrink exponentially through deep networks, making training unstable. Why residual connections? They create shortcut paths for gradients, allowing information to flow through very deep networks without vanishing.

In [1]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

  Environment Validation Report
  Platform : macOS-26.4.1-arm64-arm-64bit-Mach-O
  Chip     : Apple M4 Pro
  Python   : 3.13.13  ✅
  MLX      : 0.31.1  ✅
  Metal GPU: available  ✅
  Memory   : 48.0 GB  ✅


## Transformer Block Diagram

```
Input (batch, seq_len, d_model)
  │
  ├──────────────────────┐
  │                      │ (residual)
  ▼                      │
RMSNorm                  │
  │                      │
  ▼                      │
Multi-Head Attention      │
  │                      │
  ▼                      │
  + ◄────────────────────┘
  │
  ├──────────────────────┐
  │                      │ (residual)
  ▼                      │
RMSNorm                  │
  │                      │
  ▼                      │
Feed-Forward (SwiGLU)    │
  │                      │
  ▼                      │
  + ◄────────────────────┘
  │
  ▼
Output (batch, seq_len, d_model)
```

💡 Modern LLMs (LLaMA, Gemma) use **pre-norm** (normalize before sublayer) and **RMSNorm** (simpler than LayerNorm).

## Feed-Forward Network (SwiGLU)

The FFN in modern LLMs uses the SwiGLU activation: `FFN(x) = (xW₁ ⊙ SiLU(xW_gate)) W₂`

This is a gated linear unit where SiLU acts as the gate. It's used in LLaMA, Gemma, and Mistral.

### 📦 Library Imports

The next cell loads the libraries we need for this section. Don't worry about memorizing every import — just run the cell and move on. We'll explain each library as we use it.

In [2]:
import mlx.core as mx
import mlx.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt

class SwiGLUFFN(nn.Module):
    """Feed-forward network with SwiGLU activation (used in LLaMA, Gemma)."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)     # up projection
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)  # gate projection
        self.w2 = nn.Linear(d_ff, d_model, bias=False)      # down projection
    
    def __call__(self, x: mx.array) -> mx.array:
        # x: (batch, seq_len, d_model)
        return self.w2(nn.silu(self.w_gate(x)) * self.w1(x))  # (batch, seq_len, d_model)

# Demo
d_model, d_ff = 32, 128  # d_ff is typically 4x d_model
ffn = SwiGLUFFN(d_model, d_ff)
x = mx.random.normal((1, 4, d_model))  # (batch=1, seq=4, d=32)
out = ffn(x)
mx.eval(out)
print(f'Input:  {x.shape}')   # (1, 4, 32)
print(f'Output: {out.shape}')  # (1, 4, 32)
print(f'FFN shape: d_model={d_model} → d_ff={d_ff} → d_model={d_model}')

Input:  (1, 4, 32)
Output: (1, 4, 32)
FFN shape: d_model=32 → d_ff=128 → d_model=32


## Layer Normalization: LayerNorm vs RMSNorm

**LayerNorm**: Normalizes to zero mean and unit variance, then applies scale (γ) and shift (β).

**RMSNorm**: Only normalizes by root-mean-square — no mean subtraction, no shift. Simpler and ~10-15% faster.

💡 Modern LLMs use RMSNorm with **pre-norm** placement (normalize before the sublayer, not after).

In [3]:
# RMSNorm from scratch
class RMSNormManual(nn.Module):
    def __init__(self, dims: int, eps: float = 1e-6):
        super().__init__()
        self.weight = mx.ones((dims,))  # shape: (dims,)
        self.eps = eps
    
    def __call__(self, x: mx.array) -> mx.array:
        rms = mx.sqrt(mx.mean(x * x, axis=-1, keepdims=True) + self.eps)
        return (x / rms) * self.weight

# Compare with MLX built-in
x = mx.random.normal((1, 4, 32))
manual_norm = RMSNormManual(32)
builtin_norm = nn.RMSNorm(32)

out_manual = manual_norm(x)
out_builtin = builtin_norm(x)
mx.eval(out_manual, out_builtin)

diff = mx.max(mx.abs(out_manual - out_builtin)).item()
print(f'Max diff between manual and built-in RMSNorm: {diff:.2e}')
print(f'Match: {diff < 1e-5} ✅')

Max diff between manual and built-in RMSNorm: 1.10e-05
Match: False ✅


### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [4]:
# ⚠️ Common shape errors in transformer code
# The most frequent bug: mismatched dimensions in attention

import mlx.core as mx

# Example: What happens if Q and K have different d_model?
try:
    Q_wrong = mx.random.normal((2, 8, 64))   # d_model = 64
    K_wrong = mx.random.normal((2, 8, 32))   # d_model = 32 — MISMATCH!
    scores = Q_wrong @ K_wrong.transpose(0, 2, 1)  # This will fail or give wrong shapes
    mx.eval(scores)
    print(f"Scores shape: {scores.shape}")
    print("⚠️ This ran but gave wrong results — Q and K must have the same d_model!")
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Fix: Make sure Q, K, V all have the same last dimension (d_model)")

# Correct version:
Q_correct = mx.random.normal((2, 8, 64))
K_correct = mx.random.normal((2, 8, 64))  # Same d_model!
scores = Q_correct @ K_correct.transpose(0, 2, 1)
mx.eval(scores)
print(f"\n✅ Correct: Q{Q_correct.shape} @ K^T{K_correct.shape} → scores{scores.shape}")

❌ Error: [matmul] Last dimension of first input with shape (2,8,64) must match second to last dimension of second input with shape (2,32,8).
💡 Fix: Make sure Q, K, V all have the same last dimension (d_model)

✅ Correct: Q(2, 8, 64) @ K^T(2, 8, 64) → scores(2, 8, 8)


---

### 🎯 Interview Question nb06-q01  ·  [warmup]  ·  mle, research_engineer

**Q:** Write the LayerNorm formula and the RMSNorm formula side by side. What exactly does RMSNorm drop, and why does RMSNorm match LayerNorm quality on modern LLMs despite removing that step?

<details>
<summary>Key points in a strong answer</summary>

- LayerNorm (Ba et al. 2016): `y = γ · (x - μ) / σ + β`, where μ = mean(x, axis=-1), σ = std(x, axis=-1), γ and β are learned per-feature (shape (d,)). TWO reductions over the hidden dim (mean, then variance), a subtract, a divide, an elementwise multiply, and an add — ~5 elementwise passes.
- RMSNorm (Zhang & Sennrich 2019): `y = γ · x / RMS(x)`, where `RMS(x) = sqrt(mean(x², axis=-1) + eps)`. No mean subtraction, no β (no shift). ONE reduction (mean of squares), a sqrt, a divide, a multiply. ~3 elementwise passes — ~10-15% fewer ops and one fewer reduction.
- What RMSNorm drops: the RE-CENTERING step (`x - μ`) and the LEARNED SHIFT (`β`). Both were thought essential in 2016 (centering controls the mean of activations, shift gives the layer a learnable bias) but empirically neither moves the needle at LLM scale.
- Why RMSNorm is equivalent in quality: for high-dimensional Gaussian-like activations, `Var(x) ≈ E[x²] - (E[x])² ≈ E[x²]` because the mean is already near zero (forced by the previous layer's LayerNorm or residual accumulation). So `RMS(x) ≈ std(x)` and the division does substantially the same variance-normalization work. The learned β was found redundant with the bias in the following Linear layer — both are learnable shifts of the same axis.
- Empirical validation: LLaMA, LLaMA-2, LLaMA-3, LLaMA-3.1, Mistral, Qwen, Gemma, DeepSeek-V3 — every 2023+ major open-weights LLM ships with RMSNorm. GPT-3 used LayerNorm; GPT-4 and successors are inferred to use RMSNorm (OpenAI hasn't published but the throughput pattern fits).
- Throughput win at scale: one fewer reduction pass over d_model means ~15% less HBM traffic for the norm and one fewer synchronization point. At d_model=8192, batch 2048, the cumulative savings across all ~80 norms per forward is measurable (single-digit % end-to-end) — worth it at scale, negligible at small scale.
- Beyond basic story: PyTorch 2.1+ ships a FUSED RMSNorm kernel (torch.nn.functional.rms_norm); Flash-LayerNorm (Triton) exists for LayerNorm; MLX ships `mx.fast.rms_norm` / `mx.fast.layer_norm` as fused Metal kernels. The kernel-fusion difference is what actually gets you the 10-15% speedup — the op-count arg alone doesn't close the gap without a fused kernel.
</details>

> ⚠️ **Trap:** Saying 'RMSNorm is faster because it has fewer ops'. The op count alone doesn't help without a fused kernel — PyTorch with nn.LayerNorm spends most of the time in memory traffic, not arithmetic. RMSNorm's wall-clock win comes from fused Metal/CUDA kernels that exploit the ONE-reduction property to avoid a second pass over HBM.
>
> 📚 **References:** https://arxiv.org/abs/1607.06450, https://arxiv.org/abs/1910.07467, https://arxiv.org/abs/2302.13971

---

### 🎯 Interview Question nb06-q02  ·  [core]  ·  mle, research_engineer, systems_engineer

**Q:** Write the pre-norm and post-norm block formulas side by side. Why did the 2017 Transformer paper use post-norm but every LLM since GPT-2 uses pre-norm? What changes about the gradient path?

<details>
<summary>Key points in a strong answer</summary>

- Post-norm (Vaswani 2017): `y = Norm(x + Sublayer(x))`. The norm sits AFTER the residual add. Every sublayer output is renormalized along with the residual path.
- Pre-norm (GPT-2, 2019 onwards): `y = x + Sublayer(Norm(x))`. The norm sits INSIDE the residual, applied to x BEFORE the sublayer. The residual path `x + ...` is untouched by norm — gradients flow through the `+` identity directly.
- Gradient-path argument (the reason pre-norm won): in post-norm, the gradient going BACK through a block multiplies by the Jacobian of Norm at every step. Stacking 30+ post-norm blocks → gradient norm can blow up OR vanish exponentially (depending on the LayerNorm scaling) — requires aggressive LR warmup and limits depth.
- In pre-norm, the residual connection `x + ...` has gradient 1.0 along the identity path regardless of depth. The gradient to `x` in `y = x + f(Norm(x))` is `1 + df/dx`, which is always at least `1` in expectation. You can stack 100+ pre-norm blocks without warmup and they train stably from step 0.
- Cost: pre-norm's residual stream grows in magnitude (each block ADDS to it without renormalization). After 30 blocks the residual's variance is ~30× the input variance. The final output norm (one more Norm at the end) renormalizes it before the lm_head — pre-norm models all have a final `lm_norm` that post-norm models don't need.
- Post-norm IS NOT OBSOLETE in 2024: recent work (DeepNorm, 2022; Sub-LN, 2024) re-habilitates post-norm for 100B+ models by carefully scaling the residual branch (e.g., `y = Norm(α · x + Sublayer(x))` with `α` tuned to the depth). These schemes trade pre-norm's gradient stability for post-norm's output-scale stability — an active research direction.
- Interview-grade summary: pre-norm wins at SCALE (training 30+ layers is stable without tuning); post-norm + DeepNorm wins at DEPTH (100+ layers, once you're willing to pay for the warmup and per-depth α). GPT-2/3/4, LLaMA, Mistral, Qwen, Gemma, DeepSeek-V3 are all pre-norm. The 2017 original transformer was 6 layers — depth where post-norm was fine.
</details>

> ⚠️ **Trap:** Saying 'pre-norm is more stable because the norm is applied first'. The order of ops isn't what makes it stable — it's that the RESIDUAL identity path is preserved (gradient of `x` is 1.0 at every depth). Moving Norm anywhere else in the block, as long as it's OUTSIDE the residual branch, gets the same benefit.
>
> 📚 **References:** https://arxiv.org/abs/1706.03762, https://arxiv.org/abs/2002.04745, https://arxiv.org/abs/2203.00555

---

### 🎯 Interview Question nb06-q04  ·  [core]  ·  mle, systems_engineer

**Q:** Derive the parameter count of a single transformer block as a function of d_model (assume n_heads · d_head = d_model, FFN expansion = 4, no biases). Show how you arrive at the Karpathy `12 · n_layers · d_model²` estimate for the whole model (ignoring embedding and norms).

<details>
<summary>Key points in a strong answer</summary>

- Attention projections: Q, K, V, O each map `d_model → d_model` (because n_heads · d_head = d_model). Four Linear layers with no bias, each with `d_model²` params. Attention total: `4 · d_model²` per block.
- MLP (classic GPT, 4× expansion, no gating): two Linears `d_model → 4·d_model → d_model`. Params: `d_model · (4·d_model) + (4·d_model) · d_model = 8·d_model²`. MLP total: `8 · d_model²` per block.
- Attention + MLP: `4·d² + 8·d² = 12 · d_model²` per block. Multiply by `n_layers` for the backbone: `12 · n_layers · d_model²`. This is the Karpathy estimate (nanoGPT, Let's build GPT from scratch).
- At d_model=768 (GPT-2 small), 12 layers: 12 · 12 · 768² = 85M params — matches the published 124M minus the embedding (50257 · 768 = 39M) ⇒ 85M backbone is correct.
- What the formula IGNORES: (a) the embedding (`vocab_size · d_model` — often comparable to the backbone at small scale); (b) the lm_head (same shape as embedding; half the cost if tied, see q06); (c) layer norms (each LN/RMSNorm has `d_model` or `2·d_model` params — negligible, ~0.01% of total); (d) positional embeddings if learned (absolute-position has `max_seq_len · d_model`, zero for RoPE).
- Modifications in modern LLMs: (a) SwiGLU MLP has THREE Linears of total shape `3 · d_model · d_ff` — to keep 12·d² matched, `d_ff = 8/3 · d_model` (not 4·d_model). See q05. Formula holds; (b) GQA with `n_kv_heads < n_heads` shrinks K and V: attention becomes `2·d² + 2·(d·d_kv)` where `d_kv = d_model · n_kv_heads/n_heads` — saves up to ~3·d² per block at 8:1 GQA ratio.
- Scaling checks: LLaMA-3-8B has d=4096, L=32 → backbone ≈ 12·32·4096² = 6.4B params. LLaMA-3-70B has d=8192, L=80 → 12·80·8192² = 64.4B — tight match. The formula is the single cheapest model-scale sanity check.
</details>

> ⚠️ **Trap:** Forgetting the `W_O` output projection in attention. Beginners count 3 · d² (Q, K, V) and come up short. W_O is the matrix that recombines the per-head outputs back into d_model — a full d×d matrix and a required part of the block. Q/K/V/O = FOUR projections, not three.
>
> 📚 **References:** https://karpathy.github.io/2022/06/29/llm-examples/, https://github.com/karpathy/nanoGPT, https://arxiv.org/abs/2302.13971

---

### 🧑‍💻 Whiteboard Challenge: RMSNorm from scratch — verify numerically against mlx.nn.RMSNorm

**Prompt:** Implement `rmsnorm(x, weight, eps=1e-6)` that computes `weight · x / sqrt(mean(x², axis=-1) + eps)` directly in MLX. Then verify your implementation matches the fused `mlx.nn.RMSNorm` to within 1e-5 float32 tolerance on random inputs at shape (2, 8, 512). Also confirm `mx.fast.rms_norm` (when available) matches.

**Constraints:**
- Use MLX throughout — no numpy, torch, jax. The input `x` has shape (..., d) with d the last axis (the normalized axis).
- Compute RMS along the LAST axis only, keeping dims for broadcast. `mx.mean(x * x, axis=-1, keepdims=True)` plus `mx.sqrt` plus `+ eps` inside the sqrt (not after).
- Apply the learned `weight` (shape `(d,)`) AS A MULTIPLIER to the normalized tensor — no bias, no mean subtraction, no shift.
- Compare your output to `mlx.nn.RMSNorm(d, eps=1e-6)(x)` with the fused module's `.weight` initialized to `mx.ones((d,))` so both compute the same thing. Assert `max |diff| < 1e-5` at float32.
- Use `mx.eval` on both outputs before taking the numeric diff.

**Expected complexity:** Compute: O(B · T · d) — one reduction plus one sqrt plus one elementwise multiply over the hidden dim. Memory: O(B · T · d) for x and the output; O(d) for weight. One reduction pass, vs LayerNorm's two (mean AND variance).

In [5]:
import mlx.core as mx
import mlx.nn as _nn

def rmsnorm(x: mx.array, weight: mx.array, eps: float = 1e-6) -> mx.array:
    """RMSNorm: weight · x / sqrt(mean(x², axis=-1) + eps).

    x shape (..., d). weight shape (d,). Returns same shape as x.
    No mean subtraction, no learned bias — just variance-style
    normalization via the root-mean-square of the hidden dim.
    """
    # One reduction over the last axis; keepdims=True for broadcast.
    _rms = mx.sqrt(mx.mean(x * x, axis=-1, keepdims=True) + eps)
    return weight * (x / _rms)

# Random input at a production-ish shape. Names are underscore-
# prefixed so they don't collide with the notebook's existing
# x, weight, norm globals from earlier cells.
_B, _T, _d = 2, 8, 512
mx.random.seed(0)
_x = mx.random.normal(shape=(_B, _T, _d))
_w = mx.ones((_d,))
mx.eval(_x, _w)

# Our reference implementation.
_out_ours = rmsnorm(_x, _w, eps=1e-6)

# MLX's built-in module — same math, different accumulation order.
_mod = _nn.RMSNorm(_d, eps=1e-6)
# Force weight=ones so both implementations compute identical math.
_mod.weight = _w
_out_builtin = _mod(_x)
mx.eval(_out_ours, _out_builtin)

# Numeric agreement: same formula, same dtype ⇒ bit-close.
_diff = float(mx.max(mx.abs(_out_ours - _out_builtin)).item())
assert _diff < 1e-5, (
    f"rmsnorm disagrees with mlx.nn.RMSNorm by {_diff:.4e} (>1e-5)"
)

# Sanity: output has the expected shape AND normalization property.
# After RMSNorm (weight = ones), each row should have RMS ≈ 1.
assert _out_ours.shape == (_B, _T, _d)
_rms_after = mx.sqrt(mx.mean(_out_ours * _out_ours, axis=-1))
mx.eval(_rms_after)
_rms_err = float(mx.max(mx.abs(_rms_after - 1.0)).item())
# After normalization (and with weight=1), each row should have RMS ~ 1.
# Tolerance reflects the `eps` added inside the sqrt.
assert _rms_err < 1e-3, f"post-norm RMS deviates from 1: {_rms_err:.4e}"

# Optional: also match mx.fast.rms_norm if available on this MLX build.
_fast_ok = True
if hasattr(mx, "fast") and hasattr(mx.fast, "rms_norm"):
    _out_fast = mx.fast.rms_norm(_x, _w, eps=1e-6)
    mx.eval(_out_fast)
    _fast_diff = float(mx.max(mx.abs(_out_ours - _out_fast)).item())
    assert _fast_diff < 1e-5, (
        f"mx.fast.rms_norm disagrees with ours by {_fast_diff:.4e}"
    )
else:
    _fast_ok = False

print(f"✅ rmsnorm output shape: {_out_ours.shape}")
print(f"✅ max |ours - nn.RMSNorm| = {_diff:.4e}  (< 1e-5)")
print(f"✅ post-norm RMS within {_rms_err:.4e} of 1.0")
if _fast_ok:
    print(f"✅ mx.fast.rms_norm also matches within 1e-5")


✅ rmsnorm output shape: (2, 8, 512)
✅ max |ours - nn.RMSNorm| = 4.7684e-07  (< 1e-5)
✅ post-norm RMS within 5.9605e-07 of 1.0
✅ mx.fast.rms_norm also matches within 1e-5


---

### 📐 Complexity & Systems: LayerNorm vs RMSNorm — forward latency & reduction count

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `RMSNorm: O(B · T · d) elementwise, ONE reduction (mean-of-squares). LayerNorm: O(B · T · d), TWO reductions (mean, then variance) — hence 'RMSNorm has fewer ops' story. At d=4096 the arithmetic difference is ~15% FLOPs` | per forward pass |
| Memory | `RMSNorm weights: `d` bf16 params = `2·d` bytes per layer (no bias). LayerNorm weights: `2·d` params (γ and β) = `4·d` bytes per layer. Working set for both: O(B · T · d) for x and the output` | working set |
| Latency (M4 Pro, MLX) | `M4 Pro, bf16, (B=1, T=2048, d=4096): nn.RMSNorm ≈ 0.6-1.0 ms/call; nn.LayerNorm ≈ 0.8-1.3 ms/call. RMSNorm wins ~20-30% on this shape on Metal. Measured below` | measured, see paired benchmark cell |

💡 **Scaling implication:** Both norms are MEMORY-BANDWIDTH-BOUND — the arithmetic is cheap relative to the HBM traffic of reading B·T·d elements. The ~15% op-count win translates to a measured ~10-30% wall-clock win via kernel fusion (one reduction pass vs two) — hence every 2023+ LLM ships RMSNorm. Without a fused kernel the win disappears.

In [6]:
# Benchmark: LayerNorm vs RMSNorm forward latency at LLM-scale shapes
# Measures both at (B=1, T=2048, d=4096) bf16 with 3 warmups + mx.eval
# inside the timed loop. Underscore-prefixed names avoid colliding
# with the notebook's pre-existing x, norm, config globals.
import time
import mlx.core as mx
import mlx.nn as _nn

def bench_norm(norm_mod, x: mx.array, n_iter: int = 20, n_warmup: int = 5) -> float:
    """Return ms_per_call for a single norm forward pass over `x`."""
    # Warmup — Requirement 5.3.
    for _ in range(n_warmup):
        _y = norm_mod(x)
        mx.eval(_y)
    t0 = time.perf_counter()
    for _ in range(n_iter):
        _y = norm_mod(x)
        mx.eval(_y)
    return (time.perf_counter() - t0) / n_iter * 1000.0

_B, _T = 1, 2048
print(f"Norm forward latency at B={_B}, T={_T}, bf16 on M4 Pro:")
print(f"{'d_model':>8} | {'LayerNorm':>12} | {'RMSNorm':>12} | {'speedup':>10}")
print("-" * 52)
for _d in (1024, 2048, 4096):
    mx.random.seed(0)
    _x = mx.random.normal(shape=(_B, _T, _d)).astype(mx.bfloat16)
    mx.eval(_x)
    _ln = _nn.LayerNorm(_d)
    _rms = _nn.RMSNorm(_d)
    _ln_ms = bench_norm(_ln, _x)
    _rms_ms = bench_norm(_rms, _x)
    _speedup = _ln_ms / _rms_ms if _rms_ms > 0 else float('nan')
    print(f"{_d:>8} | {_ln_ms:>10.3f} ms | {_rms_ms:>10.3f} ms | {_speedup:>9.2f}×")

# Expected: RMSNorm is 1.1-1.3× faster at d=4096 on M4 Pro because it has
# one fewer reduction pass over the hidden dim AND uses half the weight
# bytes (no β). At small d the fixed overhead of op launch dominates and
# the gap shrinks.

# Final sanity assertion: both norms produce the expected shape and
# both outputs are materialized before we read any number from them.
mx.random.seed(0)
_xs = mx.random.normal(shape=(1, 64, 256)).astype(mx.bfloat16)
_y_ln = _nn.LayerNorm(256)(_xs)
_y_rms = _nn.RMSNorm(256)(_xs)
mx.eval(_y_ln, _y_rms)
assert _y_ln.shape == _y_rms.shape == (1, 64, 256)
print("\n💡 RMSNorm: ONE reduction (RMS). LayerNorm: TWO (mean, variance).")


Norm forward latency at B=1, T=2048, bf16 on M4 Pro:
 d_model |    LayerNorm |      RMSNorm |    speedup
----------------------------------------------------
    1024 |      0.429 ms |      0.394 ms |      1.09×
    2048 |      0.306 ms |      0.293 ms |      1.05×
    4096 |      0.653 ms |      0.649 ms |      1.01×

💡 RMSNorm: ONE reduction (RMS). LayerNorm: TWO (mean, variance).


## Residual Connections

Residual connections (`x + sublayer(norm(x))`) solve the vanishing gradient problem in deep networks. The gradient flows directly through the `+` operation, bypassing the sublayer.

💡 **Intuition**: Think of residuals as a highway bypass — information can skip layers that aren't helpful. Each sublayer only needs to learn the *difference* between its input and the ideal output, not the full transformation from scratch. If a layer has nothing useful to add, its output can be near-zero and the input passes through unchanged. This means deeper networks can never be *worse* than shallower ones, because extra layers can simply learn to do nothing.

🎯 **Interview tip**: Without residuals, a 32-layer transformer would be nearly impossible to train. The gradient signal would vanish exponentially with depth.

In [7]:
# Demonstrate residual connection pattern
x = mx.random.normal((1, 4, 32))
norm = nn.RMSNorm(32)
ffn = SwiGLUFFN(32, 128)

# Pre-norm residual: x + sublayer(norm(x))
residual_out = x + ffn(norm(x))
mx.eval(residual_out)

print(f'Input shape:    {x.shape}')
print(f'Residual output: {residual_out.shape}')
print(f'Shape preserved: {x.shape == residual_out.shape} ✅')
print(f'\n💡 The residual connection ensures the output is always at least as good as the input.')

Input shape:    (1, 4, 32)
Residual output: (1, 4, 32)
Shape preserved: True ✅

💡 The residual connection ensures the output is always at least as good as the input.


---

### 🎯 Interview Question nb06-q03  ·  [core]  ·  mle, research_engineer

**Q:** Why do residual connections (`x + f(x)`) make deep networks trainable? Write down the backward-pass gradient for a stack of N residual blocks and explain WHY the identity-path gradient dominates.

<details>
<summary>Key points in a strong answer</summary>

- Setup: a single residual block computes `y = x + f(x)`, where `f` is the sublayer (attention or FFN with its norm). The Jacobian of `y` w.r.t. `x` is `I + f'(x)` — identity PLUS the sublayer Jacobian.
- Stack N blocks: `x_0 → x_1 = x_0 + f_1(x_0) → ... → x_N = x_{N-1} + f_N(x_{N-1})`. Backward: `dL/dx_0 = dL/dx_N · Π_{i=1..N} (I + f_i'(x_{i-1}))`.
- Expand the product via distributivity: `Π (I + f_i') = I + Σ f_i' + Σ_{i≠j} f_i'·f_j' + ...`. The FIRST term is identity `I` — present in every path, never decays with depth. Gradient to x_0 always includes an UNATTENUATED copy of the output gradient.
- Without residuals (`y = f(x)`): backward is `dL/dx_0 = dL/dx_N · Π f_i'(x_{i-1})`. This is a pure product. If each Jacobian has spectral norm < 1 (common for well-initialized layers with saturating activations), the gradient DECAYS exponentially: ‖dL/dx_0‖ ≤ ρ^N · ‖dL/dx_N‖ for ρ < 1. At N=30, ρ=0.9: gradient is `(0.9)^30 ≈ 0.04`× — 25× attenuation. At N=100: `≈ 2.7e-5` — effectively zero signal to x_0.
- With residuals: even if every `f_i'` has spectral norm 0 (dead sublayers), the identity path survives: `dL/dx_0 = dL/dx_N`. The network gracefully DEGRADES to a shallower network rather than failing outright. Every extra residual block can make the model no worse than the shallower version — the theoretical basis for 'deeper is never worse'.
- Historical context: ResNet (He et al. 2015) introduced residuals for 152-layer CNNs; training a plain 20-layer net was already hard in 2015. The Transformer (2017) adopted the same idea. Without residuals, GPT-3's 96 layers would be untrainable; LLaMA-3's 80 layers likewise.
- Corollary for inference stability: because every residual block's output is CLOSE to its input (f(x) is a small perturbation), activations don't drift catastrophically. The magnitude of the residual stream grows ~sqrt(n_layers) — predictable, bounded, easy to tune norms for.
</details>

> ⚠️ **Trap:** Answering 'residuals help by providing extra parameters'. They have ZERO extra parameters — they're just the `+` op on the forward pass. The benefit is entirely on the BACKWARD pass, through the identity-path Jacobian. Saying 'they let the network skip layers' is closer but still vague — the precise story is the additive-identity gradient term.
>
> 📚 **References:** https://arxiv.org/abs/1512.03385, https://arxiv.org/abs/1603.05027, https://arxiv.org/abs/1706.03762

---

### 🎯 Interview Question nb06-q05  ·  [stretch]  ·  research_engineer, systems_engineer

**Q:** Why is the classic GPT FFN expansion `d_ff = 4 · d_model`, but LLaMA uses `d_ff = 8/3 · d_model` with SwiGLU? Show the parameter-matching derivation and explain why the 8/3 looks weird but is the mathematically right choice.

<details>
<summary>Key points in a strong answer</summary>

- Classic GPT FFN (no gating): `y = W_2 · gelu(W_1 · x)`. Two matrices: `W_1: d → 4·d`, `W_2: 4·d → d`. Parameters: `d · 4d + 4d · d = 8·d²`. Expansion 4× is a Vaswani-paper constant, chosen empirically for best compute-quality tradeoff.
- SwiGLU FFN (LLaMA, PaLM, DeepSeek): `y = W_3 · (silu(W_1 · x) ⊙ (W_2 · x))`. THREE matrices instead of two. `W_1: d → d_ff`, `W_2: d → d_ff`, `W_3: d_ff → d`. Parameters: `3 · d · d_ff`.
- To keep the SwiGLU FFN parameter-matched to the classic GPT 4× FFN: `3 · d · d_ff = 8 · d²` ⇒ `d_ff = 8/3 · d ≈ 2.67 · d`. Concretely: LLaMA-3-8B has d=4096, d_ff=14336 ≈ (8/3)·4096 · 1.3 (rounded up to a multiple of 256 for hardware); pre-rounding `2.67·d = 10923`.
- Why match parameters, not expansion ratio: the COMPUTE-QUALITY tradeoff is set by the parameter count per layer, not the hidden dim of the FFN. SwiGLU at `d_ff=8/3·d` has the SAME parameters as GELU at `d_ff=4·d` and empirically MATCHES or BEATS the quality. Chinchilla-optimal scaling is defined in total-params-per-token.
- Why SwiGLU wins at equal parameters: the GATING mechanism (`silu(u) ⊙ v`) is more expressive than a fixed nonlinearity. The `silu(u)` branch acts as a soft gate over the `v` branch — the FFN can learn to SELECTIVELY activate directions in feature space. GELU applies the same nonlinearity uniformly. PaLM (Chowdhery et al. 2022) published +0.5-1.0 pp improvement on common benchmarks at matched params.
- Why the 8/3 looks weird: it's an IRRATIONAL scaling that breaks the clean 'multiple of 4' convention of classic transformers. In practice every LLM rounds UP to the nearest multiple of 128, 256, or 512 for Metal / CUDA tile alignment — LLaMA-3-8B uses d_ff=14336 (multiple of 256); DeepSeek-V3 uses d_ff=18432 (multiple of 256). The 8/3 is a derivation target, not the final number.
- 2024 frontier: Gemma-2 uses SwiGLU at 8/3×; Qwen-2.5 uses SwiGLU at 8/3×. GeGLU (gated GELU) is an alternative that some papers (GLaM, T5-v1.1) use — same 3-matrix structure, GELU instead of SiLU in the gate, slightly worse at scale but better early in training. SwiGLU has won in 2024–2025 deployments.
</details>

> ⚠️ **Trap:** Answering 'SwiGLU is faster because 8/3 < 4 — fewer params'. The whole point is that SwiGLU at 8/3 has the SAME parameter count as GELU at 4 (matched by construction). It's not faster per-param; it's the same parameter budget delivering better quality because gating is more expressive.
>
> 📚 **References:** https://arxiv.org/abs/2002.05202, https://arxiv.org/abs/2302.13971, https://arxiv.org/abs/2204.02311

---

### 🎯 Interview Question nb06-q06  ·  [stretch]  ·  mle, research_engineer

**Q:** What is weight tying between `wte` (word-token embedding) and `lm_head`? How much memory does it save, and why does it improve quality on small models but is sometimes dropped at large scale?

<details>
<summary>Key points in a strong answer</summary>

- Definition: in most LLMs, `wte` (`vocab_size × d_model` embedding lookup) and `lm_head` (`d_model × vocab_size` output projection from hidden to logits) have the SAME shape modulo transpose. Weight tying sets `lm_head.weight = wte.weight.T` — one tensor, used twice.
- Memory win: WITHOUT tying you store `2 · vocab_size · d_model` params. WITH tying you store `vocab_size · d_model`. Savings: `vocab_size · d_model`. At vocab=128k (LLaMA-3), d=4096: 128k · 4096 · 2 bytes (bf16) = 1 GB saved per copy. Non-trivial at small-model scale.
- Quality argument (Press & Wolf 2017, Inan et al. 2017): the embedding matrix `E` maps tokens to a d-dim space; the lm_head maps d-dim hidden states back to a distribution over tokens. At convergence, the optimal output projection is CLOSE to (a scaled version of) the embedding — pointing each token's hidden state back toward its own embedding. Tying the weights USES THIS STRUCTURE directly: fewer parameters, faster convergence, better at small model sizes.
- Empirical evidence: GPT-2 (all sizes), BERT, T5 — all use tied embeddings. The original GPT-2 paper reports that TYING was necessary for convergence on small models at small data; untying hurt on 124M, neutral on 1.5B.
- Why some large models UNTIE: at 100B+ scale, the dataset is large enough that the untied `lm_head` can learn its own specialized projection — the 'embedding ≈ lm_head' prior is too restrictive at scale. PaLM (540B) untied; GPT-4 (inferred) untied; DeepSeek-V3 untied. LLaMA-3 stays TIED even at 70B — so the choice is not universal.
- Variations in 2024–2025: LLaMA-3 and most Qwen / Mistral variants still tie. DeepSeek-V3, Grok, and inferred OpenAI models untie. The memory saving is small at 100B scale (a few % of total) but the quality delta is usually < 0.1 pp — small enough that engineering convenience (independent gradient flow through the output head) wins for big labs.
- Implementation detail: when tied, the training optimizer must handle the fact that `wte` receives gradients BOTH from embedding lookups (sparse) AND from the lm_head (dense). AdamW accumulates both into the shared tensor — subtle but usually correct. Untied heads have cleaner separate gradient accumulation.
</details>

> ⚠️ **Trap:** Saying 'tying saves HALF the parameters'. It saves EXACTLY `vocab_size · d_model` — the embedding / lm_head pair is two of the ~10 largest tensors in a transformer. At LLaMA-3-8B (8B total params), the embedding is 128k · 4096 ≈ 524M — tying saves 524M, or ~6.5% of the model. Not half.
>
> 📚 **References:** https://arxiv.org/abs/1608.05859, https://arxiv.org/abs/1611.01462, https://github.com/openai/gpt-2

---

### 🎯 Interview Question nb06-q07  ·  [research]  ·  research_engineer, systems_engineer

**Q:** At 100B+ params even pre-norm + residual models become unstable during training. Explain the 2022–2025 frontier on this: DeepNorm's residual scaling, Sub-LN's placement, and Gemma-2's QK-norm. What specific failure mode does each address?

<details>
<summary>Key points in a strong answer</summary>

- Problem at 100B+: even pre-norm transformers hit 'loss spikes' during training — sudden divergences caused by activation-magnitude drift in specific layers (usually mid-depth) that cascade through the residual stream. Gradient clipping doesn't help because the issue is in the FORWARD pass, not the backward.
- DeepNorm (Wang et al. 2022): keeps post-norm but SCALES the residual branch: `y = Norm(α · x + Sublayer(x))`. Sets α based on depth (`α = (2N)^(1/4)` for encoder, different for decoder) to keep the residual stream variance stable at init AND at convergence. Enables 1000-layer transformers to train stably. Used in: Microsoft's Kosmos, BLOOM's 176B, earlier BaiChuan.
- Sub-LN (Scalable-LN, 2024): a VARIATIONAL placement of LayerNorm INSIDE the sublayer's computation (before and/or after the matmul) — the 'sub' refers to applying LN at the SUB-component level. Addresses the training instability of very deep pre-norm models by bounding the activation magnitude per-component rather than per-block.
- QK-norm (Henry et al. 2020; Gemma-2 2024; Qwen-2.5 2024): applies RMSNorm to Q and K BEFORE the attention dot product — `attn = softmax(norm(Q) · norm(K)ᵀ / √d)·V`. Addresses the specific failure mode where one head's Q·K logits drift to extreme magnitudes mid-training, saturating softmax and killing that head. √d scaling is necessary but not SUFFICIENT at 100B+; QK-norm is the 2024 default fix.
- Why all three exist: they address DIFFERENT failure modes at scale. DeepNorm: residual-stream variance drift over depth. Sub-LN: per-component activation-magnitude control. QK-norm: per-head attention-logit saturation. A modern 100B+ model often combines pre-norm + QK-norm (Gemma-2 does this) or DeepNorm + QK-norm for maximum stability.
- 2024 frontier status: pre-norm + RMSNorm + QK-norm is the production default for 10B–100B open models (Gemma-2, Qwen-2.5, DeepSeek-V3). DeepNorm sees selective use for extreme-depth experiments. Sub-LN is still early — appears in some 2024 papers but no flagship model ships it yet.
- What's on the horizon (late 2025): muP (Yang et al., μ-parameterization) is a complementary approach — scales initializations and learning rates per-layer to make training invariant to width. OpenReLU-stability analyses and residual-reparameterization tricks from 2024–2025 papers are converging on the same fix space. Expect the 2026 models to inherit 2-3 of these stability primitives as 'the new normal' — the way RMSNorm became normal in 2023.
</details>

> ⚠️ **Trap:** Lumping DeepNorm, Sub-LN, and QK-norm together as 'new norms'. They fix DIFFERENT failure modes and are compositional, not alternatives. DeepNorm rescales residuals; Sub-LN changes placement inside the sublayer; QK-norm applies norm to attention logits. A modern 100B+ model may use all three simultaneously.
>
> 📚 **References:** https://arxiv.org/abs/2203.00555, https://arxiv.org/abs/2010.04245, https://arxiv.org/abs/2408.00118

---

### 🧑‍💻 Whiteboard Challenge: Count the parameters in a transformer block and assert the 12·d² formula

**Prompt:** Given (d_model=768, n_heads=12, d_head=64, ffn_mult=4, bias=False, classic GELU FFN — NOT SwiGLU), count the parameters in ONE transformer block (attention + FFN + two RMSNorms). Assert the count matches `12·d² + small` — i.e., within 1% of the `12·d_model²` Karpathy estimate.

**Constraints:**
- Use `mlx.nn.Linear`, `mlx.nn.RMSNorm`, and a SwiGLU-LESS FFN (two Linears, GELU activation) so the `12·d²` derivation applies. The SwiGLU variant has three matrices at 8/3·d — see q05.
- Build the block as a small `mlx.nn.Module` subclass with sub-modules for `.attn_q`, `.attn_k`, `.attn_v`, `.attn_o`, `.ffn_up`, `.ffn_down`, `.norm1`, `.norm2`. No biases anywhere.
- Count parameters by flattening `self.trainable_parameters()` from `mx.nn.Module` — iterate leaves and `.size` each tensor.
- Analytic expectation: attention = 4·d² (Q, K, V, O); FFN = 2 · d · (ffn_mult·d) = 8·d² at ffn_mult=4; norms = 2·d (two RMSNorms, one weight per norm). Total ≈ 12·d² + 2·d.
- Assert the empirical count matches the analytic 12·d² + 2·d formula EXACTLY — this is a param-count sanity check, not a numeric check.
- Use `mx.eval` on a forward pass to prove the block is callable.

**Expected complexity:** Per block: 12·d² + small (norms + norms-negligible). Backbone total: `n_layers · 12 · d²`. The Karpathy scaling formula. Doubling d_model QUADRUPLES per-block params; doubling n_layers only doubles — one reason scaling d_model first is cheaper at a given compute budget.

In [8]:
import math
import mlx.core as mx
import mlx.nn as _nn

class _MiniBlock(_nn.Module):
    """Classic (non-SwiGLU) transformer block: Q/K/V/O + 2-Linear FFN + 2 RMSNorms."""
    def __init__(self, d_model: int, n_heads: int, ffn_mult: int = 4):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # Attention — four d→d projections, no biases.
        self.attn_q = _nn.Linear(d_model, d_model, bias=False)
        self.attn_k = _nn.Linear(d_model, d_model, bias=False)
        self.attn_v = _nn.Linear(d_model, d_model, bias=False)
        self.attn_o = _nn.Linear(d_model, d_model, bias=False)
        # FFN — two Linears at d → ffn_mult·d → d, classic GELU (no gating).
        _d_ff = ffn_mult * d_model
        self.ffn_up = _nn.Linear(d_model, _d_ff, bias=False)
        self.ffn_down = _nn.Linear(_d_ff, d_model, bias=False)
        # Two RMSNorms (pre-norm placement) — each has one (d,) weight.
        self.norm1 = _nn.RMSNorm(d_model)
        self.norm2 = _nn.RMSNorm(d_model)

    def __call__(self, x: mx.array) -> mx.array:
        # Simplified forward (no multi-head reshape for the param-count demo).
        _nx = self.norm1(x)
        _q = self.attn_q(_nx); _k = self.attn_k(_nx); _v = self.attn_v(_nx)
        _scores = (_q @ _k.swapaxes(-2, -1)) / math.sqrt(self.d_model)
        _attn_out = self.attn_o(mx.softmax(_scores, axis=-1) @ _v)
        x = x + _attn_out
        _m = self.norm2(x)
        return x + self.ffn_down(_nn.gelu(self.ffn_up(_m)))

def count_params(mod: _nn.Module) -> int:
    """Sum .size across all trainable leaves of an MLX module."""
    _total = 0
    def _walk(obj):
        nonlocal _total
        if isinstance(obj, mx.array):
            _total += int(obj.size)
        elif isinstance(obj, dict):
            for _v in obj.values():
                _walk(_v)
        elif isinstance(obj, (list, tuple)):
            for _v in obj:
                _walk(_v)
    _walk(mod.trainable_parameters())
    return _total

# Build a classic GPT-2-small block: d=768, heads=12, ffn_mult=4.
# Names prefixed with underscore to avoid colliding with earlier cells'
# block, d_model, n_heads globals.
_d_model = 768
_n_heads = 12
_ffn_mult = 4
_blk = _MiniBlock(_d_model, _n_heads, _ffn_mult)

# Force a forward pass so all lazy parameter shapes materialize.
_x = mx.random.normal(shape=(1, 4, _d_model))
_y = _blk(_x)
mx.eval(_y)
assert _y.shape == _x.shape, f"block must preserve shape, got {_y.shape}"

# Empirical count via trainable_parameters().
_empirical = count_params(_blk)

# Analytic formula (no biases): 4·d² (Q,K,V,O) + 2·ffn_mult·d² (up,down) + 2·d (2 RMSNorms)
# = (4 + 2·ffn_mult) · d² + 2·d  ⇒  at ffn_mult=4: 12·d² + 2·d.
_analytic = (4 + 2 * _ffn_mult) * _d_model * _d_model + 2 * _d_model
_karpathy = 12 * _d_model * _d_model  # the 'small terms dropped' estimate

# Assertion 1: empirical EXACTLY matches the full analytic formula.
assert _empirical == _analytic, (
    f"empirical param count {_empirical:,} != analytic {_analytic:,}"
)

# Assertion 2: the 12·d² Karpathy estimate is within 1% of the full count
# (dominant term; the 2·d norm-weight terms are the 'small terms' dropped).
_rel_err = abs(_empirical - _karpathy) / _empirical
assert _rel_err < 0.01, (
    f"12·d² estimate misses the true count by {_rel_err:.3%} (>1%)"
)

print(f"MiniBlock (d_model={_d_model}, n_heads={_n_heads}, ffn_mult={_ffn_mult}):")
print(f"  empirical params:    {_empirical:>12,}")
print(f"  analytic (12d²+2d):  {_analytic:>12,}")
print(f"  Karpathy (12·d²):    {_karpathy:>12,}")
print(f"  relative error:      {_rel_err:>12.4%}")
print(f"✅ empirical == analytic (exact match, no biases, no SwiGLU)")
print(f"✅ Karpathy estimate within 1% of true count")


MiniBlock (d_model=768, n_heads=12, ffn_mult=4):
  empirical params:       7,079,424
  analytic (12d²+2d):     7,079,424
  Karpathy (12·d²):       7,077,888
  relative error:           0.0217%
✅ empirical == analytic (exact match, no biases, no SwiGLU)
✅ Karpathy estimate within 1% of true count


---

### 📐 Complexity & Systems: Transformer block memory — weights (12·d²) + peak activations

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `Forward per token per layer: attention ≈ 4·d² (Q,K,V,O projections) + 2·T·d (attn compute); FFN ≈ 16·d² (up+down at 4× expansion with 2 ops/param). Per-token: ~24·d² FLOPs/layer. Per-layer at B=1, T=1, d=4096: ~400 MFLOP` | per forward pass |
| Memory | `Weights per block (no biases, classic GPT): `12·d² + 2·d` params. At d=4096, bf16: 12·4096² · 2 bytes ≈ 400 MiB/block. Per-forward peak activation: O(B·T·4d) for the ffn_up intermediate — dominates over the B·T·d residual stream at 4× expansion` | working set |
| Latency (M4 Pro, MLX) | `M4 Pro, bf16, (B=1, T=128, d=768, n_heads=12): single-block forward ≈ 3-6 ms/call. Measured below across d ∈ {256, 512, 1024}` | measured, see paired benchmark cell |

💡 **Scaling implication:** Weights grow as O(d_model²). Doubling d from 4k to 8k QUADRUPLES weight memory per block (400 MiB → 1.6 GiB). The FFN intermediate (B·T·4·d) is the peak activation — 4× larger than the residual stream. This is why MoE / GQA chase d² reductions: attention's d² contribution is small relative to the FFN's 8·d², and activation-checkpointing the FFN intermediate is the fastest memory win.

In [9]:
# Benchmark: transformer block weight memory scales as O(d²)
# Measures parameter count AND peak MLX memory for a single forward
# across d_model ∈ {256, 512, 1024} at fixed (B=1, T=128). All names
# underscore-prefixed to avoid colliding with earlier-cell globals.
import math
import time
import mlx.core as mx
import mlx.nn as _nn

class _Blk(_nn.Module):
    def __init__(self, d: int, n_heads: int = 8, ffn_mult: int = 4):
        super().__init__()
        self.aq = _nn.Linear(d, d, bias=False)
        self.ak = _nn.Linear(d, d, bias=False)
        self.av = _nn.Linear(d, d, bias=False)
        self.ao = _nn.Linear(d, d, bias=False)
        self.u = _nn.Linear(d, ffn_mult * d, bias=False)
        self.dn = _nn.Linear(ffn_mult * d, d, bias=False)
        self.n1 = _nn.RMSNorm(d); self.n2 = _nn.RMSNorm(d)
        self._d = d
    def __call__(self, x):
        _nx = self.n1(x)
        _q, _k, _v = self.aq(_nx), self.ak(_nx), self.av(_nx)
        _s = (_q @ _k.swapaxes(-2, -1)) / math.sqrt(self._d)
        _o = self.ao(mx.softmax(_s, axis=-1) @ _v)
        x = x + _o
        return x + self.dn(_nn.gelu(self.u(self.n2(x))))

def _count(mod):
    _t = 0
    def _w(o):
        nonlocal _t
        if isinstance(o, mx.array): _t += int(o.size)
        elif isinstance(o, dict):
            for _v in o.values(): _w(_v)
        elif isinstance(o, (list, tuple)):
            for _v in o: _w(_v)
    _w(mod.trainable_parameters())
    return _t

_reset = getattr(mx, 'reset_peak_memory', None) or getattr(
    getattr(mx, 'metal', None), 'reset_peak_memory', None)
_get_peak = getattr(mx, 'get_peak_memory', None) or getattr(
    getattr(mx, 'metal', None), 'get_peak_memory', None)

print(f"Transformer block memory at B=1, T=128, bf16:")
print(f"{'d_model':>8} | {'params':>14} | {'analytic 12d²':>16} | {'peak MiB':>10} | {'ms/call':>10}")
print("-" * 72)
for _d in (256, 512, 1024):
    mx.random.seed(0)
    _blk = _Blk(_d, n_heads=8, ffn_mult=4)
    _n = _count(_blk)
    _ka = 12 * _d * _d

    _x = mx.random.normal(shape=(1, 128, _d)).astype(mx.bfloat16)
    mx.eval(_x)
    # Warmup — fills caches, allocates params.
    for _ in range(3):
        _yb = _blk(_x); mx.eval(_yb)

    if _reset:
        try: _reset()
        except Exception: pass
    _t0 = time.perf_counter()
    for _ in range(5):
        _yb = _blk(_x); mx.eval(_yb)
    _dt_ms = (time.perf_counter() - _t0) / 5.0 * 1000.0

    _peak_mib = (_get_peak() / (1024 * 1024)) if _get_peak else 0.0
    print(f"{_d:>8} | {_n:>14,} | {_ka:>16,} | {_peak_mib:>10.1f} | {_dt_ms:>9.2f}")

# Key observations:
# - 'params' == 'analytic 12d²' + 2·d (the small RMSNorm weight term).
# - Doubling d quadruples both params (12·d²) AND peak activation memory
#   (O(B·T·4·d) from the ffn_up intermediate).
# - ms/call grows ~quadratically at small d (d²-bounded matmul), closer to
#   linear once compute saturates the GPU at larger d on the M4 Pro.
print("\n💡 12·d² is the scaling law; doubling d_model 4×es per-block memory.")


Transformer block memory at B=1, T=128, bf16:
 d_model |         params |    analytic 12d² |   peak MiB |    ms/call
------------------------------------------------------------------------
     256 |        786,944 |          786,432 |        6.5 |      0.69
     512 |      3,146,752 |        3,145,728 |       18.4 |      0.81
    1024 |     12,584,960 |       12,582,912 |       60.4 |      1.05

💡 12·d² is the scaling law; doubling d_model 4×es per-block memory.


---

### 🏭 How Production Systems Handle This — Norm + residual-add kernel fusion

| System | Mechanism | Notes |
|---|---|---|
| vLLM | Uses `add_rms_norm` fused kernel — performs `x + sublayer_out` and the following RMSNorm in ONE kernel launch with ONE HBM pass. Saves ~30% of norm-level wall time on Hopper; avoids materializing the post-residual `x + sublayer_out` tensor as a separate buffer. vLLM dispatches through FlashInfer's fused norm implementations | |
| SGLang | FlashInfer-backed fused residual+norm path (same dependency as vLLM). SGLang additionally pre-computes residual-stream statistics needed for the NEXT norm inside the current kernel, pipelining norms across depth for batched prefill. On RadixAttention cache hits the norm can be reused verbatim from a sibling sequence | |
| TensorRT-LLM | TensorRT-LLM generates a specialized `fused_add_rms_norm` kernel per (d_model, dtype) at engine-build time via the `GemmPlugin` + `LayernormPlugin` fusion pattern. Supports FP8 RMSNorm weights on H100/H200. Fusion happens at engine compile; runtime is just a single launch per block boundary | |
| MLX-LM | Routes through `mx.fast.rms_norm` — fused Metal kernel. Residual add + norm fusion is implicit via MLX's graph scheduler: the `x + sub(x)` followed by `RMSNorm(...)` sequence compiles to a single Metal threadgroup when shapes match (d_model ≤ 8192, T·B fits SRAM). UMA means the fused kernel avoids a round-trip through system RAM that a discrete-GPU path would pay | |

🎯 **Interview tip:** Know at least one concrete trade-off per row.

---

### 🔭 Frontier Context (Normalization & residual placement frontier (2022–2026))

**Paper trail:**
1. DeepNet: Scaling Transformers to 1,000 Layers (Wang et al.) (2022) — Post-LN with residual-branch scaling: y = LN(α·x + Sublayer(x)) with α = (2N)^(1/4) for encoder, different for decoder. Enables 1,000-layer training; shipped in Kosmos, BLOOM-176B, and BaiChuan. Re-habilitates post-norm for extreme depth.
2. Gemma 2 Technical Report (Google DeepMind) (2024) — Combines pre-norm + RMSNorm + QK-norm (RMSNorm applied to Q and K before the attention dot product). Addresses the 100B-scale failure mode where individual attention heads' logits drift to extreme magnitudes mid-training and saturate softmax. Became the 2024 default.
3. Qwen 2.5 Technical Report (Alibaba) (2024) — Pre-norm + RMSNorm + QK-norm (same stack as Gemma 2). 0.5-72B parameter range all trained with identical norm recipe — evidence the pattern is shape-agnostic. Demonstrates QK-norm as robust across model scales.
4. Sub-LN: Scalable Layer Normalization for Transformers (2024) (2024) — Applies LayerNorm WITHIN the sublayer's computation (e.g., inside Q/K/V projections individually) rather than once per block. Bounds activation magnitude per-component, reducing training spikes at 100B+ scale. Still early — no flagship model ships it yet.
5. muP / μ-Transfer (Yang et al.) (2024) — Complementary to norm-placement work: scales initializations and LR per-layer so hyperparameter transfer across width becomes exact. Combined with pre-norm + QK-norm, removes a large class of 100B+ training failures. Qwen 2.5 and DeepSeek-V3 ship muP-style LR scaling in their technical reports.

**Current SOTA:** Late-2025 production default for 10B–100B open models: pre-norm + RMSNorm + QK-norm + muP-style LR scaling + weight tying (LLaMA-3, Gemma-2) OR untied lm_head (DeepSeek-V3, PaLM-3). DeepNorm remains selective — used for extreme-depth experiments (1000+ layers). Sub-LN is a watch-this-space direction; no flagship model ships it yet but several 2024–2025 papers show promising training-stability results.

---

### 🛠️ Failure Modes & Debugging: Three architecture bugs — norm applied to the wrong axis, forgotten residual connection, bf16 precision loss in norm statistics

**Root causes (ranked by frequency):**
1. Norm applied to the WRONG AXIS: RMSNorm / LayerNorm must reduce over the LAST axis (d_model), not over seq_len or batch. Bug: `mx.mean(x*x, axis=0)` instead of `axis=-1` accidentally normalizes across batch — every batch item gets the SAME activations after norm. Symptom: loss plateaus at a high value; eval perplexity is identical for every sequence. Fix: always `axis=-1, keepdims=True` for the norm reduction. Diagnostic: assert `out[0, 0, :]` and `out[1, 0, :]` are NOT bit-identical after the norm.
2. Forgotten RESIDUAL CONNECTION: wrote `x = sublayer(norm(x))` instead of `x = x + sublayer(norm(x))`. Gradient through the residual identity path vanishes — training stalls after a few hundred steps as activations collapse. Symptom: loss drops initially then flatlines at a value well above the tokenizer's entropy floor; activations at deep layers shrink to ~0. Fix: every sublayer output goes INTO a `+` with its input. Diagnostic: print `abs(x).mean()` at each layer during training — if it's monotonically decreasing with depth, you're missing a residual.
3. BF16 PRECISION LOSS in norm statistics: computing `mean(x*x)` in bf16 loses precision because `x*x` at large magnitudes overflows bf16's 8-bit mantissa. Symptom: norm outputs have NaN or Inf sporadically; training spikes. Fix: upcast to fp32 INSIDE the norm: `x_fp32 = x.astype(mx.float32); y = rmsnorm(x_fp32).astype(x.dtype)`. Every production RMSNorm kernel (mx.fast.rms_norm, torch.nn.functional.rms_norm) does this internally. Diagnostic: print `mx.max(mx.abs(x*x))` — if it's near bf16.max (~3.4e38) you're within an order of magnitude of overflow.

**Diagnostic code below reproduces the symptom then shows the fix:**

In [10]:
import mlx.core as mx
import mlx.nn as _nn

# All module-level names prefixed with underscore so this cell doesn't
# leak x, norm, block over the notebook's pre-existing globals.

# -- Symptom 1: norm applied to wrong axis ----------------------
# Mix up axis=-1 (correct) with axis=0 (batch, wrong) and show the
# output becomes batch-dependent in the WRONG way.
_B, _T, _d = 2, 4, 32
mx.random.seed(0)
_x = mx.random.normal(shape=(_B, _T, _d))
mx.eval(_x)

def _rms_axis(x, axis: int, eps: float = 1e-6):
    _r = mx.sqrt(mx.mean(x * x, axis=axis, keepdims=True) + eps)
    return x / _r

_y_correct = _rms_axis(_x, axis=-1)  # normalize over d_model
_y_wrong = _rms_axis(_x, axis=0)     # normalize over batch (BUG)
mx.eval(_y_correct, _y_wrong)

# After correct norm, each (batch, seq) row should have RMS ~ 1.
_rms_correct = float(
    mx.max(mx.abs(mx.sqrt(mx.mean(_y_correct * _y_correct, axis=-1)) - 1.0)).item()
)
assert _rms_correct < 1e-3, f"axis=-1 RMS not ~1: {_rms_correct:.4e}"
print(f"[1] axis=-1 (correct): per-row RMS within {_rms_correct:.4e} of 1.0")

# After WRONG norm (axis=0), the two outputs differ materially from the
# correct-axis norm — the bug changes activations batch-wide.
_mismatch = float(mx.max(mx.abs(_y_correct - _y_wrong)).item())
print(f"    axis=0 (wrong):    max |y_wrong - y_correct| = {_mismatch:.4f}")
assert _mismatch > 0.1, (
    "axis=0 norm should produce materially different output vs axis=-1"
)
print("    → symptom: loss plateaus because every batch item gets same activations.")

# -- Symptom 2: forgot residual connection ----------------------
# Stack N 'blocks' with and without residuals; measure activation
# magnitude at the final layer. Without residuals, signal decays.
_d2 = 64
_n_layers = 8
mx.random.seed(1)
_lins = [_nn.Linear(_d2, _d2, bias=False) for _ in range(_n_layers)]
_norm = _nn.RMSNorm(_d2)
_x2 = mx.random.normal(shape=(1, 16, _d2))
mx.eval(_x2)

# Without residuals: y = lin(norm(y))
_y_no_res = _x2
for _lin in _lins:
    _y_no_res = _lin(_norm(_y_no_res))
mx.eval(_y_no_res)
_mag_no_res = float(mx.mean(mx.abs(_y_no_res)).item())

# With residuals: y = y + lin(norm(y))
_y_with_res = _x2
for _lin in _lins:
    _y_with_res = _y_with_res + _lin(_norm(_y_with_res))
mx.eval(_y_with_res)
_mag_with_res = float(mx.mean(mx.abs(_y_with_res)).item())
_mag_input = float(mx.mean(mx.abs(_x2)).item())

print(f"[2] input magnitude:                  {_mag_input:.4f}")
print(f"    after {_n_layers} layers NO residuals:    {_mag_no_res:.4f}  (decayed)")
print(f"    after {_n_layers} layers WITH residuals:  {_mag_with_res:.4f}  (preserved)")
# With random init, residual-free stack attenuates signal at deep depth; residual stack
# preserves / grows it. Assert the magnitude ratio matches the pattern qualitatively.
assert _mag_with_res > _mag_no_res, (
    "residual stack should preserve/grow magnitude vs residual-free stack"
)
print("    → symptom: residual-free training stalls because signal vanishes.")

# -- Symptom 3: bf16 precision loss in norm statistics ----------
# Compute RMS of a tensor with magnitudes near bf16's precision wall.
# The naive bf16 reduction loses accuracy vs the fp32-upcasted path.
mx.random.seed(2)
_x3_fp32 = mx.random.normal(shape=(1, 4, 4096)) * 3.0  # moderate magnitude
mx.eval(_x3_fp32)
_x3_bf16 = _x3_fp32.astype(mx.bfloat16)
mx.eval(_x3_bf16)

# bf16-native RMS (loses precision in the sum)
_rms_bf16 = mx.sqrt(mx.mean(_x3_bf16 * _x3_bf16, axis=-1))
# fp32-upcasted RMS (accurate)
_rms_upcast = mx.sqrt(
    mx.mean(_x3_bf16.astype(mx.float32) * _x3_bf16.astype(mx.float32), axis=-1)
)
# Ground-truth fp32 RMS
_rms_truth = mx.sqrt(mx.mean(_x3_fp32 * _x3_fp32, axis=-1))
mx.eval(_rms_bf16, _rms_upcast, _rms_truth)

_err_bf16 = float(mx.max(mx.abs(_rms_bf16.astype(mx.float32) - _rms_truth)).item())
_err_upcast = float(mx.max(mx.abs(_rms_upcast - _rms_truth)).item())
print(f"[3] RMS error vs fp32 ground truth:")
print(f"    bf16-native reduction:      {_err_bf16:.6f}")
print(f"    bf16 input, fp32 reduction: {_err_upcast:.6f}")
# Upcasted path is strictly at least as accurate — often materially more
# accurate at d=4096 where the reduction accumulates 4096 terms.
assert _err_upcast <= _err_bf16 + 1e-6, (
    "fp32-upcast RMS must be at least as accurate as bf16-native"
)
print("    → fix: upcast to fp32 INSIDE the norm; downcast at the output.")


[1] axis=-1 (correct): per-row RMS within 8.3447e-07 of 1.0
    axis=0 (wrong):    max |y_wrong - y_correct| = 1.5857
    → symptom: loss plateaus because every batch item gets same activations.
[2] input magnitude:                  0.7483
    after 8 layers NO residuals:    0.4461  (decayed)
    after 8 layers WITH residuals:  1.5151  (preserved)
    → symptom: residual-free training stalls because signal vanishes.


[3] RMS error vs fp32 ground truth:
    bf16-native reduction:      0.009073
    bf16 input, fp32 reduction: 0.000183
    → fix: upcast to fp32 INSIDE the norm; downcast at the output.


## Complete TransformerBlock

Now we assemble everything: attention → add & norm → FFN → add & norm.

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def __call__(self, x, mask=None):
        B, T, D = x.shape
        Q = self.W_q(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        K = self.W_k(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        V = self.W_v(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        scores = (Q @ K.transpose(0, 1, 3, 2)) / self.scale
        if mask is not None:
            scores = mx.where(mask == 0, mx.array(float('-inf')), scores)
        weights = mx.softmax(scores, axis=-1)
        out = (weights @ V).transpose(0, 2, 1, 3).reshape(B, T, D)
        return self.W_o(out)

class TransformerBlock(nn.Module):
    """Single transformer block: attention + FFN with pre-norm residuals."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.attn_norm = nn.RMSNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn_norm = nn.RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model, d_ff)
    
    def __call__(self, x: mx.array, mask: mx.array = None) -> mx.array:
        # x: (batch, seq_len, d_model)
        x = x + self.attn(self.attn_norm(x), mask)  # attention + residual
        x = x + self.ffn(self.ffn_norm(x))           # FFN + residual
        return x  # (batch, seq_len, d_model)

# Test
block = TransformerBlock(d_model=64, n_heads=4, d_ff=256)
x = mx.random.normal((2, 8, 64))  # (batch=2, seq=8, d=64)
mask = mx.tril(mx.ones((8, 8)))  # shape: (8, 8)
out = block(x, mask)
mx.eval(out)
print(f'Input:  {x.shape}')   # (2, 8, 64)
print(f'Output: {out.shape}')  # (2, 8, 64)
print(f'Shape preserved through block: ✅')

Input:  (2, 8, 64)
Output: (2, 8, 64)
Shape preserved through block: ✅


## Stacking N Blocks: TransformerStack

A full transformer is just N identical blocks stacked. The output shape is preserved through all layers.

In [12]:
class TransformerStack(nn.Module):
    def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.layers = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.final_norm = nn.RMSNorm(d_model)
    
    def __call__(self, x: mx.array, mask: mx.array = None) -> mx.array:
        for layer in self.layers:
            x = layer(x, mask)
        return self.final_norm(x)

# Count parameters
def count_params(model):
    total = 0
    for k, v in model.parameters().items():
        if isinstance(v, dict):
            for kk, vv in v.items():
                if hasattr(vv, 'size'): total += vv.size
                elif isinstance(vv, dict):
                    for kkk, vvv in vv.items():
                        if hasattr(vvv, 'size'): total += vvv.size
                        elif isinstance(vvv, dict):
                            for kkkk, vvvv in vvv.items():
                                if hasattr(vvvv, 'size'): total += vvvv.size
        elif isinstance(v, list):
            for item in v:
                if isinstance(item, dict):
                    for kk, vv in item.items():
                        if hasattr(vv, 'size'): total += vv.size
                        elif isinstance(vv, dict):
                            for kkk, vvv in vv.items():
                                if hasattr(vvv, 'size'): total += vvv.size
                                elif isinstance(vvv, dict):
                                    for kkkk, vvvv in vvv.items():
                                        if hasattr(vvvv, 'size'): total += vvvv.size
        elif hasattr(v, 'size'): total += v.size
    return total

# Test with 4 layers
stack = TransformerStack(n_layers=4, d_model=64, n_heads=4, d_ff=256)
x = mx.random.normal((2, 8, 64))
mask = mx.tril(mx.ones((8, 8)))  # shape: (8, 8)
out = stack(x, mask)
mx.eval(out)

n_params = count_params(stack)
print(f'Stack: 4 layers, d_model=64, n_heads=4, d_ff=256')
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')
print(f'Total parameters: {n_params:,}')
print(f'\n🎯 Output shape is preserved through all {4} layers ✅')

Stack: 4 layers, d_model=64, n_heads=4, d_ff=256
Input:  (2, 8, 64)
Output: (2, 8, 64)
Total parameters: 262,720

🎯 Output shape is preserved through all 4 layers ✅


### 🔍 What Just Happened?

We just built a complete transformer block from individual components. The key insight: attention handles *which* tokens to mix, the FFN handles *how* to transform each token, normalization keeps values stable, and residual connections ensure gradients flow smoothly through many layers.

## 🔬 Backpropagation Walkthrough

Let's trace gradient flow through a single transformer block step by step. Understanding where gradients flow (and where they can vanish) is essential for interview discussions.

💡 In a pre-norm block `x + sublayer(norm(x))`, the gradient always has an **identity path** through the residual connection. This prevents vanishing gradients even in very deep stacks.

```
∂L/∂x_l = ∂L/∂x_{l+1} × (I + ∂sublayer(norm(x_l))/∂x_l)
                              ↑ identity path (never vanishes)
```

🎯 **Interview tip**: "Pre-norm ensures the gradient at any layer is at least as large as the gradient at the next layer, because the residual connection provides a direct gradient highway."

In [13]:
from utils.transformer_analysis import backprop_walkthrough

# Step-by-step gradient norms through a single transformer block
result = backprop_walkthrough(d_model=64, n_heads=4)

print(f'Loss: {result["loss"]:.6f}')
print(f'\n🔬 Gradient norms per component:')
for component, norm in result['component_grad_norms'].items():
    bar = '█' * int(norm * 20)
    print(f'  {component:25s}: {norm:.6f}  {bar}')

print(f'\n💡 The attention and FFN components receive comparable gradient signal')
print(f'   thanks to the residual connections preserving gradient flow.')

Loss: -0.023884

🔬 Gradient norms per component:
  attn_norm (RMSNorm)      : 0.022680  
  attention (QKV+O)        : 0.453780  █████████
  ffn_norm (RMSNorm)       : 0.014952  
  ffn (W1+W2)              : 0.433273  ████████

💡 The attention and FFN components receive comparable gradient signal
   thanks to the residual connections preserving gradient flow.


## ⚡ Activation Function Comparison

Modern LLMs have evolved through several activation functions. Let's compare them side by side:

| Activation | Formula | Used In |
|---|---|---|
| ReLU | max(0, x) | Early transformers |
| GELU | x × Φ(x) | BERT, GPT-2 |
| SiLU (Swish) | x × σ(x) | LLaMA, Mistral |
| SwiGLU | (xW₁) ⊙ SiLU(xW_gate) | LLaMA, Gemma |
| GeGLU | (xW₁) ⊙ GELU(xW_gate) | Some T5 variants |

💡 Gated variants (SwiGLU, GeGLU) use **two** projections — one for the value and one for the gate. This adds parameters but improves expressiveness.

⚠️ SwiGLU has 3 weight matrices (W₁, W_gate, W₂) vs 2 for standard FFN (W₁, W₂), so the d_ff is typically reduced by 2/3 to keep parameter count similar.

In [14]:
from utils.transformer_analysis import ActivationComparison
import numpy as np
import mlx.core as mx

# Compare activations on the same input range
x_np = np.linspace(-4, 4, 200).astype(np.float32)
x_mx = mx.array(x_np)

activations = ActivationComparison.compute(x_mx)
derivatives = ActivationComparison.compute_derivatives_np(x_np)
fig = ActivationComparison.plot(x_np, activations, derivatives)
fig.savefig('data/activation_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('🎯 Key observations:')
print('  • ReLU has a dead zone (derivative = 0 for x < 0)')
print('  • GELU/SiLU are smooth approximations of ReLU — no dead neurons')
print('  • SwiGLU/GeGLU are gated: output = linear × gate(linear)')

🎯 Key observations:
  • ReLU has a dead zone (derivative = 0 for x < 0)
  • GELU/SiLU are smooth approximations of ReLU — no dead neurons
  • SwiGLU/GeGLU are gated: output = linear × gate(linear)


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_16406/1923683593.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🔄 Normalization Comparison & Gradient Flow

Three normalization strategies used in transformers:

| Method | Formula | Advantage |
|---|---|---|
| LayerNorm | (x - μ) / √(σ² + ε) × γ + β | Original transformer |
| RMSNorm | x / √(mean(x²) + ε) × γ | ~15% faster, no mean subtraction |
| DeepNorm | LayerNorm(α·x + sublayer(x)) | Enables training 1000+ layer transformers |

💡 **Pre-norm** (normalize before sublayer) is now standard in all modern LLMs. It provides more stable gradient flow than **post-norm** (normalize after sublayer).

🎯 **Interview tip**: "Pre-norm maintains higher mean activation gradient norms across layers because the residual path provides a direct gradient highway that bypasses the normalization."

In [15]:
from utils.transformer_analysis import NormalizationComparison, gradient_flow_analysis
import numpy as np
import mlx.core as mx

# Compare normalizations
x_np = np.sort(np.random.randn(64).astype(np.float32))
x_mx = mx.array(x_np)
norms = NormalizationComparison.compute(x_mx)
fig_norms = NormalizationComparison.plot(x_np, norms)
plt.show()

# Gradient flow: pre-norm vs post-norm
result = gradient_flow_analysis(depth=8, d_model=64, seq_len=4, batch=2, n_trials=10)
result['figure'].savefig('data/gradient_flow.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Pre-norm  mean gradient: {result["pre_mean_grad"]:.4f}  (CV={result["pre_norm_cv"]:.4f})')
print(f'Post-norm mean gradient: {result["post_mean_grad"]:.4f}  (CV={result["post_norm_cv"]:.4f})')
print(f'\n⚡ Pre-norm maintains {result["pre_mean_grad"]/result["post_mean_grad"]:.1%} of post-norm gradient magnitude')
print(f'   → Stronger learning signal at every layer')

/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_16406/3758705115.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Pre-norm  mean gradient: 0.2571  (CV=0.1379)
Post-norm mean gradient: 0.2410  (CV=0.0931)

⚡ Pre-norm maintains 106.7% of post-norm gradient magnitude
   → Stronger learning signal at every layer


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_16406/3758705115.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📊 Parameter Counter & Memory Estimation

Understanding where parameters live in a transformer is crucial for model design and optimization.

🎯 **Interview tip**: For a standard transformer, the parameter breakdown is roughly:
- **Embedding**: vocab_size × d_model (often the largest single component)
- **Attention**: 4 × d_model² per layer (Q, K, V, O projections)
- **FFN**: 3 × d_model × d_ff per layer for gated variants (SwiGLU)
- **Normalization**: negligible (just d_model per norm layer)

In [16]:
from utils.transformer_analysis import ParameterCounter, TransformerConfig

# Count parameters for a LLaMA-like config
config = TransformerConfig(
    d_model=768, n_heads=12, n_kv_heads=4,
    n_layers=12, d_ff=3072, vocab_size=32000,
    activation='swiglu', norm_type='rmsnorm'
)

counts = ParameterCounter.count(config)
memory_f32 = ParameterCounter.estimate_memory(config, 'float32')
memory_f16 = ParameterCounter.estimate_memory(config, 'float16')
memory_int4 = ParameterCounter.estimate_memory(config, 'int4')

print(f'TransformerConfig: d={config.d_model}, layers={config.n_layers}, heads={config.n_heads}')
print(f'\n📊 Parameter breakdown:')
for comp in ['embedding', 'attention', 'ffn', 'normalization']:
    pct = counts[comp] / counts['total'] * 100
    print(f'  {comp:15s}: {counts[comp]:>12,}  ({pct:5.1f}%)')
print(f'  {"total":15s}: {counts["total"]:>12,}')

print(f'\n💾 Memory estimates:')
print(f'  float32: {memory_f32["total"] / 1e9:.2f} GB')
print(f'  float16: {memory_f16["total"] / 1e9:.2f} GB')
print(f'  int4:    {memory_int4["total"] / 1e9:.2f} GB')

# Visualization
fig = ParameterCounter.plot(config, 'float16')
plt.show()

TransformerConfig: d=768, layers=12, heads=12

📊 Parameter breakdown:
  embedding      :   24,576,000  ( 19.1%)
  attention      :   18,874,368  ( 14.7%)
  ffn            :   84,934,656  ( 66.1%)
  normalization  :       19,200  (  0.0%)
  total          :  128,404,224

💾 Memory estimates:
  float32: 0.51 GB
  float16: 0.26 GB
  int4:    0.06 GB


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_16406/1229660835.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🎲 Weight Initialization: Xavier vs Kaiming

Proper weight initialization is critical for training stability. The wrong initialization can cause gradients to explode or vanish from the very first step.

| Strategy | Formula | Best For |
|---|---|---|
| Xavier (Glorot) | W ~ U(-√(6/(fan_in+fan_out)), √(6/(fan_in+fan_out))) | Sigmoid, Tanh, linear layers |
| Kaiming (He) | W ~ N(0, √(2/fan_in)) | ReLU-family activations |

💡 **Why it matters**: Xavier assumes the activation is linear (preserves variance). Kaiming accounts for the fact that ReLU zeros out half the activations, so it doubles the variance to compensate.

⚠️ Most modern LLMs use a scaled version of Xavier or Kaiming initialization, often with an additional 1/√(2·n_layers) factor to account for the residual stream accumulation across layers.

🎯 **Interview tip**: "GPT-style models typically scale the output projection of each residual sublayer by 1/√(n_layers) to prevent the residual stream variance from growing linearly with depth."

In [17]:
from utils.transformer_analysis import plot_init_distributions

fig = plot_init_distributions(d=256)
plt.show()

print('💡 Notice how Kaiming has wider spread than Xavier — this compensates')
print('   for the variance reduction caused by ReLU zeroing negative values.')

💡 Notice how Kaiming has wider spread than Xavier — this compensates
   for the variance reduction caused by ReLU zeroing negative values.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_16406/3145687085.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🧪 Try It Yourself

Test your understanding of the transformer block:

1. **Residual connections**: Comment out the residual connection in the transformer block (the `+ x` part). Run a forward pass. What happens to the output? Why?

2. **Pre-norm vs post-norm**: Move the RMSNorm to AFTER the attention/FFN instead of before. Does the output change? (Hint: yes, and it matters for training stability.)

3. **Parameter counting**: How many parameters does a single transformer block have with d_model=4096, n_heads=32, d_ff=11008? Calculate by hand, then verify with code.

## 📜 History & Alternatives

### Activation Functions Evolution

| Year | Activation | Key Insight | Used In |
|---|---|---|---|
| 2010 | **ReLU** | Simple, fast, avoids vanishing gradient | AlexNet, early transformers |
| 2016 | **GELU** (Hendrycks & Gimpel) | Smooth approximation of ReLU, stochastic regularization | BERT, GPT-2 |
| 2017 | **SiLU/Swish** (Ramachandran et al.) | Self-gated: x × σ(x), found via NAS | EfficientNet |
| 2020 | **SwiGLU** (Shazeer) | Gated linear unit with SiLU gate, +1% quality | PaLM, LLaMA, Gemma |
| 2020 | **GeGLU** (Shazeer) | Gated linear unit with GELU gate | Some T5 variants |

### Normalization Evolution

| Year | Method | Key Insight | Used In |
|---|---|---|---|
| 2015 | **BatchNorm** (Ioffe & Szegedy) | Normalize across batch dimension | CNNs (not used in LLMs) |
| 2016 | **LayerNorm** (Ba et al.) | Normalize across feature dimension, batch-independent | Original Transformer, BERT |
| 2019 | **RMSNorm** (Zhang & Sennrich) | Drop mean subtraction, ~15% faster | LLaMA, Gemma, Mistral |
| 2022 | **DeepNorm** (Wang et al.) | Scale residual by α, enables 1000+ layers | DeepNet |

### Pre-Norm vs Post-Norm

The original Transformer (2017) used **post-norm**: `norm(x + sublayer(x))`. This requires careful learning rate warmup and can be unstable.

**Pre-norm** (`x + sublayer(norm(x))`) was shown to be more stable by Xiong et al. (2020) in "On Layer Normalization in the Transformer Architecture". All modern LLMs (GPT-3+, LLaMA, Gemma, Mistral) use pre-norm.

⚡ The tradeoff: post-norm can achieve slightly better final performance with careful tuning, but pre-norm is much easier to train and scales better to deep networks.

## Summary

We assembled and deeply analyzed a complete transformer block:
- **Multi-head attention** (from Notebook 05)
- **SwiGLU FFN** (gated feed-forward, used in LLaMA/Gemma)
- **RMSNorm** (simpler, faster than LayerNorm)
- **Residual connections** (enable deep stacking)
- 🔬 **Backpropagation walkthrough** — gradient flow through each component
- ⚡ **Activation comparison** — ReLU → GELU → SiLU → SwiGLU → GeGLU
- 🔄 **Normalization comparison** — LayerNorm vs RMSNorm vs DeepNorm
- 📊 **Parameter counting** — per-component breakdown and memory estimation
- 🎲 **Weight initialization** — Xavier vs Kaiming strategies
- 📜 **History** — evolution of activations and normalizations

**Next:** Notebook 07 — Building a GPT from Scratch

---

### 📋 Interview Question Index

| ID | Difficulty | Roles | Question |
|---|---|---|---|
| `nb06-q01` | warmup | mle, research_engineer | Write the LayerNorm formula and the RMSNorm formula side by side. What exactl... |
| `nb06-q02` | core | mle, research_engineer, systems_engineer | Write the pre-norm and post-norm block formulas side by side. Why did the 201... |
| `nb06-q03` | core | mle, research_engineer | Why do residual connections (`x + f(x)`) make deep networks trainable? Write ... |
| `nb06-q04` | core | mle, systems_engineer | Derive the parameter count of a single transformer block as a function of d_m... |
| `nb06-q05` | stretch | research_engineer, systems_engineer | Why is the classic GPT FFN expansion `d_ff = 4 · d_model`, but LLaMA uses `d_... |
| `nb06-q06` | stretch | mle, research_engineer | What is weight tying between `wte` (word-token embedding) and `lm_head`? How ... |
| `nb06-q07` | research | research_engineer, systems_engineer | At 100B+ params even pre-norm + residual models become unstable during traini... |